# Lila ASI v2 vs `llama-cpp-python` — Google Colab CPU Benchmark

This notebook benchmarks **Lila ASI v2 (`liblila_asi.so`)** against **`llama-cpp-python`** on Google Colab CPU hardware (typically Intel Xeon with **AVX-512F / AVX-512BW / AVX-512VNNI** or AMD EPYC with **AVX2 / AVX-512**).

### Before running
1. Upload a tarball of your `lila` source (`source/engine` + `source/src/core/asi_backend.py`) **or** set `GIT_REPO_URL` below.
   - To create a minimal tarball locally in WSL:
     ```bash
     cd /home/ticket/workspaces/lila
     tar -czf lila_asi_colab.tar.gz source/engine source/src/core/asi_backend.py
     ```
2. Make sure Runtime is set to **CPU** (or High-RAM CPU).

In [ ]:
#@title 1. Inspect Colab CPU & SIMD Capabilities
!lscpu | grep -E 'Model name|CPU\(s\)|Thread|Core|Socket|Flags' | head -n 10
import re
flags = set(re.findall(r'\S+', open('/proc/cpuinfo').read().split('flags')[1].split('\n')[0]))
for f in ['avx2', 'fma', 'f16c', 'avx_vnni', 'avx512f', 'avx512bw', 'avx512vl', 'avx512_vnni']:
    print(f"{f:14s}: {'YES' if f in flags else 'no'}")

In [ ]:
#@title 2. Load Lila ASI Source (Git clone OR upload lila_asi_colab.tar.gz)
import os
GIT_REPO_URL = ""  #@param {type:"string"}

os.makedirs("/content/lila", exist_ok=True)
%cd /content/lila

if GIT_REPO_URL:
    !git clone --depth 1 "{GIT_REPO_URL}" /content/lila_repo
    !cp -r /content/lila_repo/source /content/lila/
elif os.path.exists("/content/lila_asi_colab.tar.gz"):
    !tar -xzf /content/lila_asi_colab.tar.gz -C /content/lila
else:
    from google.colab import files
    print("Upload lila_asi_colab.tar.gz (created via: tar -czf lila_asi_colab.tar.gz source/engine source/src/core/asi_backend.py)")
    uploaded = files.upload()
    fname = next(iter(uploaded.keys()))
    !tar -xzf "{fname}" -C /content/lila

!ls -la /content/lila/source/engine

In [ ]:
#@title 3. Build liblila_asi.so with -O3 -march=native (enables AVX-512 / AVX-VNNI if supported)
%cd /content/lila/source/engine
!make clean && make lila-engine lila-asi liblila-asi test-v2
!ls -lh asi/liblila_asi.so

In [ ]:
#@title 4. Install Python dependencies (llama-cpp-python, huggingface_hub)
!pip install -q --upgrade huggingface_hub
!CMAKE_ARGS="-DGGML_NATIVE=ON" pip install -q llama-cpp-python --no-binary llama-cpp-python || pip install -q llama-cpp-python

In [ ]:
#@title 5. Download Qwen3.5-0.8B GGUF & Convert to .asi v2 Cartridge
from huggingface_hub import hf_hub_download
import os

HF_REPO = "mradermacher/Qwen3.5-0.8B-heretic-i1-GGUF"  #@param {type:"string"}
HF_FILE = "Qwen3.5-0.8B-heretic.i1-Q4_K_M.gguf"  #@param {type:"string"}

gguf_path = hf_hub_download(repo_id=HF_REPO, filename=HF_FILE, local_dir="/content/models")
asi_path = "/content/qwen35_0.8b.asi"
print("Downloaded:", gguf_path)

!python3 /content/lila/source/engine/asi/gguf_to_asi_v2.py "{gguf_path}" "{asi_path}"
!ls -lh "{asi_path}"

In [ ]:
#@title 6. Run Apples-to-Apples Benchmark (Lila ASI v2 vs llama.cpp)
import os, sys, time
sys.path.insert(0, "/content/lila/source/src")

from core.asi_backend import AsiLlamaAdapter
from llama_cpp import Llama

N_THREADS = os.cpu_count() or 2  #@param {type:"integer"}
MAX_TOKENS = 40  #@param {type:"integer"}

MESSAGES = [
    {"role": "system", "content": "You are Lila, a concise local AI assistant."},
    {"role": "user", "content": "Give me a 1-sentence greeting and explain what 2+2 equals."},
]

def bench(name, loader):
    t0 = time.monotonic()
    llm = loader()
    load_s = time.monotonic() - t0
    print(f"\n=== {name} (threads={N_THREADS}) ===")
    print(f"Cold load: {load_s:.2f}s")
    for turn in (1, 2):
        t1 = time.monotonic()
        ttft = None
        parts = []
        for ev in llm.create_chat_completion(
            messages=MESSAGES, max_tokens=MAX_TOKENS, temperature=0.0, top_p=1.0, top_k=1, stream=True
        ):
            delta = (ev.get("choices") or [{}])[0].get("delta", {}).get("content")
            if delta:
                if ttft is None:
                    ttft = time.monotonic() - t1
                parts.append(delta)
        total_s = time.monotonic() - t1
        text = "".join(parts)
        gen_s = max(1e-6, total_s - (ttft or total_s))
        tok_s = max(0, MAX_TOKENS - 1) / gen_s if parts else 0.0
        print(f"Turn {turn}: TTFT={ttft:.2f}s | total={total_s:.2f}s | decode={tok_s:.2f} tok/s")
        print(f"  Output: {text.strip()[:160]}")

bench("Lila ASI v2", lambda: AsiLlamaAdapter(asi_path, n_ctx=512, n_threads=N_THREADS, verbose=False))
bench("llama.cpp", lambda: Llama(model_path=gguf_path, n_ctx=512, n_threads=N_THREADS, verbose=False))